# 🌐 Global AI Layoffs — Data Cleaning & Preprocessing
**Dataset:** [Kaggle – Global AI Layoffs and Job Market 2020–Present](https://www.kaggle.com/datasets/belbino/global-ai-layoffs-and-job-market-2020-present)


---

## 📌 Business Goals

| # | Goal | KPIs |
|---|------|------|
| 1 | Which industries were most affected? | Total layoffs by industry, avg layoffs per company |
| 2 | How did layoffs trend over time? | Quarterly trends, year-over-year changes |
| 3 | Are AI companies more resilient? | Avg % workforce laid off (AI vs non-AI) |
| 4 | Which cities/countries had the highest reductions? | Total layoffs by country, top 10 cities |
| 5 | Which funding stages are most vulnerable? | Avg layoffs by stage, layoff event frequency |

---

## 🔧 Cleaning Steps
1. Import libraries & load data  
2. Initial inspection  
3. Handle missing values  
4. Fix data types  
5. Standardize text fields  
6. Feature engineering  
7. Final validation & export

In [ ]:
import pandas as pd
import numpy as np
import os
import zipfile

## 1. Load Data

In [ ]:
# --- NEW TOKEN SETUP ---
# Replace 'YOUR_KAGGLE_USERNAME' with your actual Kaggle username
os.environ['KAGGLE_USERNAME'] = "amgadyassin"

# Paste the long secure token string you copied from the 'Layoff_Project' menu here
os.environ['KAGGLE_KEY'] = "KGAT_63524ddf79e566c389b1974616a2164b"
# -----------------------

# Now we safely import the Kaggle API (it will automatically look at the environment variables above)
from kaggle.api.kaggle_api_extended import KaggleApi

def fetch_automated_layoffs_df():
    api = KaggleApi()
    api.authenticate()

    dataset_slug = "belbino/global-ai-layoffs-and-job-market-2020-present"
    target_folder = "../data/"
    target_file = "layoffs_events.csv"

    # Create the folder if it doesn't exist yet
    os.makedirs(target_folder, exist_ok=True)

    print("Connecting to Kaggle using 'Layoff_Project' token...")
    api.dataset_download_files(dataset_slug, path=target_folder, unzip=False)

    zip_path = os.path.join(target_folder, "global-ai-layoffs-and-job-market-2020-present.zip")
    extracted_file_path = os.path.join(target_folder, target_file)

    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        if target_file in zip_ref.namelist():
            zip_ref.extract(target_file, target_folder)
            print(f"Successfully extracted: {target_file}")
        else:
            print(f"Error: {target_file} not found in zip.")
            import sys
            sys.exit(1) # Clean exit using sys instead of the bare exit()

    if os.path.exists(zip_path):
        os.remove(zip_path)

    df = pd.read_csv(extracted_file_path)
    return df

df = fetch_automated_layoffs_df()

Connecting to Kaggle using 'Layoff_Project' token...
Dataset URL: https://www.kaggle.com/datasets/belbino/global-ai-layoffs-and-job-market-2020-present
Successfully extracted: layoffs_events.csv


In [21]:
print(f"Shape: {df.shape}")
df.head()

Shape: (2470, 11)


,company,location,layoff_count,date,pct_workforce,industry,source_url,stage,raised_mm,country,is_ai_company
0,Panda Squad,SF Bay Area,6.0,2020-03-13,75%,Consumer,https://twitter.com/danielsing er/status/12385...,Seed,$1.00,United States,False
1,HopSkipDrive,Los Angeles,8.0,2020-03-13,10%,Transportat…,https://layoffs.fyi/2020/04/02/ hopskipdrive-l...,Unknown,$45.00,United States,False
2,Help.com,Austin,16.0,2020-03-16,100%,Support,LinkedIn,Seed,$6.00,United States,False
3,Service,Los Angeles,NaN,2020-03-16,100%,Travel,https://techcrunch.com/2020/ 03/16/travel-savi...,Seed,$5.00,United States,False
4,Inspirato,Denver,130.0,2020-03-16,22%,Travel,https://businessden.com/2020 /03/16/inspirato-...,Series C,$79.00,United States,False


## 2. Initial Inspection

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2470 entries, 0 to 2469
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   company        2470 non-null   str    
 1   location       2470 non-null   str    
 2   layoff_count   1592 non-null   float64
 3   date           2470 non-null   str    
 4   pct_workforce  1534 non-null   str    
 5   industry       2468 non-null   str    
 6   source_url     2469 non-null   str    
 7   stage          2465 non-null   str    
 8   raised_mm      2179 non-null   str    
 9   country        2468 non-null   str    
 10  is_ai_company  2470 non-null   bool   
dtypes: bool(1), float64(1), str(9)
memory usage: 195.5 KB


In [22]:
df.describe()

,layoff_count
count,1592.000000
mean,340.008166
std,1397.407383
min,4.000000
25%,40.000000
50%,89.500000
75%,200.000000
max,30000.000000


In [23]:
df.dtypes

company              str
location             str
layoff_count     float64
date                 str
pct_workforce        str
industry             str
source_url           str
stage                str
raised_mm            str
country              str
is_ai_company       bool
dtype: object

In [5]:
missing = df.isnull().sum().reset_index()
missing.columns = ["Column", "Missing Count"]
missing["Missing %"] = (missing["Missing Count"] / len(df) * 100).round(2)
missing[missing["Missing Count"] > 0].sort_values("Missing %", ascending=False)

,Column,Missing Count,Missing %
4,pct_workforce,936,37.89
2,layoff_count,878,35.55
8,raised_mm,291,11.78
7,stage,5,0.20
5,industry,2,0.08
9,country,2,0.08
6,source_url,1,0.04


## 3. Handle Missing Values

**Strategy:**
- `layoff_count`: Flag missing as `layoff_count_missing`, then impute with industry median
- `pct_workforce`: Strip `%` symbol, convert to float, flag missing, impute with industry median
- `raised_mm`: Strip `$` and whitespace, convert to float, flag missing, impute with stage median
- `industry` / `country` / `stage`: Fill with `"Unknown"`

In [7]:
df["layoff_count_missing"] = df["layoff_count"].isnull().astype(bool)
df["raised_mm_missing"]    = df["raised_mm"].isnull().astype(bool)
print("Missing flags created ✅")

Missing flags created ✅


## 4. Fix Data Types

In [8]:
df["pct_workforce"] = (
    df["pct_workforce"]
    .astype(str)
    .str.replace("%", "", regex=False)
    .str.strip()
    .replace("nan", np.nan)
    .astype(float)
)
print("pct_workforce → float ✅")
df["pct_workforce"].describe()

pct_workforce → float ✅


count    1534.000000
mean       30.026728
std        30.510421
min         0.000000
25%        10.000000
50%        19.000000
75%        33.000000
max       100.000000
Name: pct_workforce, dtype: float64

In [9]:
df["raised_mm"] = (
    df["raised_mm"]
    .astype(str)
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
    .replace("nan", np.nan)
    .astype(float)
)
print("raised_mm → float ✅")
df["raised_mm"].describe()

raised_mm → float ✅


count      2179.000000
mean        964.510785
std        5691.260382
min           0.000000
25%          50.500000
50%         171.000000
75%         482.500000
max      121900.000000
Name: raised_mm, dtype: float64

In [10]:
df["date"] = pd.to_datetime(df["date"], errors="coerce")
print("date → datetime ✅")
df["date"].dtype

date → datetime ✅


dtype('<M8[us]')

## 5. Standardize Text Fields

In [11]:
text_cols = ["company", "location", "industry", "stage", "country"]

for col in text_cols:
    df[col] = df[col].astype(str).str.strip().str.title()

# Fill remaining unknowns
df["industry"] = df["industry"].replace("Nan", "Unknown")
df["country"]  = df["country"].replace("Nan", "Unknown")
df["stage"]    = df["stage"].replace("Nan", "Unknown")

print("Text columns standardized ✅")
df[text_cols].head()

Text columns standardized ✅


,company,location,industry,stage,country
0,Panda Squad,Sf Bay Area,Consumer,Seed,United States
1,Hopskipdrive,Los Angeles,Transportat…,Unknown,United States
2,Help.Com,Austin,Support,Seed,United States
3,Service,Los Angeles,Travel,Seed,United States
4,Inspirato,Denver,Travel,Series C,United States


## 6. Impute Missing Values

In [12]:
industry_median_layoffs = df.groupby("industry")["layoff_count"].median()
df["layoff_count"] = df.apply(
    lambda row: industry_median_layoffs.get(row["industry"], df["layoff_count"].median())
    if pd.isnull(row["layoff_count"]) else row["layoff_count"],
    axis=1
)
print(f"layoff_count nulls remaining: {df['layoff_count'].isnull().sum()} ✅")

layoff_count nulls remaining: 0 ✅


In [13]:
industry_median_pct = df.groupby("industry")["pct_workforce"].median()
df["pct_workforce"] = df.apply(
    lambda row: industry_median_pct.get(row["industry"], df["pct_workforce"].median())
    if pd.isnull(row["pct_workforce"]) else row["pct_workforce"],
    axis=1
)
print(f"pct_workforce nulls remaining: {df['pct_workforce'].isnull().sum()} ✅")

pct_workforce nulls remaining: 0 ✅


In [14]:
stage_median_raised = df.groupby("stage")["raised_mm"].median()
df["raised_mm"] = df.apply(
    lambda row: stage_median_raised.get(row["stage"], df["raised_mm"].median())
    if pd.isnull(row["raised_mm"]) else row["raised_mm"],
    axis=1
)
print(f"raised_mm nulls remaining: {df['raised_mm'].isnull().sum()} ✅")

raised_mm nulls remaining: 0 ✅


## 7. Feature Engineering

In [15]:
df["year"]    = df["date"].dt.year
df["month"]   = df["date"].dt.month
df["quarter"] = df["date"].dt.quarter
df["year_quarter"] = df["date"].dt.to_period("Q").astype(str)

print("Date features created ✅")
df[["date", "year", "month", "quarter", "year_quarter"]].head()

Date features created ✅


,date,year,month,quarter,year_quarter
0,2020-03-13,2020,3,1,2020Q1
1,2020-03-13,2020,3,1,2020Q1
2,2020-03-16,2020,3,1,2020Q1
3,2020-03-16,2020,3,1,2020Q1
4,2020-03-16,2020,3,1,2020Q1


## 8. Final Validation & Export

In [16]:
print("=== Final Shape ===")
print(df.shape)

print("\n=== Remaining Nulls ===")
print(df.isnull().sum()[df.isnull().sum() > 0])

print("\n=== Data Types ===")
print(df.dtypes)

df.head()

=== Final Shape ===
(2470, 17)

=== Remaining Nulls ===
industry      2
source_url    1
stage         5
country       2
dtype: int64

=== Data Types ===
company                            str
location                           str
layoff_count                   float64
date                    datetime64[us]
pct_workforce                  float64
industry                           str
source_url                         str
stage                              str
raised_mm                      float64
country                            str
is_ai_company                     bool
layoff_count_missing              bool
raised_mm_missing                 bool
year                             int32
month                            int32
quarter                          int32
year_quarter                       str
dtype: object


,company,location,layoff_count,date,pct_workforce,industry,source_url,stage,raised_mm,country,is_ai_company,layoff_count_missing,raised_mm_missing,year,month,quarter,year_quarter
0,Panda Squad,Sf Bay Area,6.0,2020-03-13,75.0,Consumer,https://twitter.com/danielsing er/status/12385...,Seed,1.0,United States,False,False,False,2020,3,1,2020Q1
1,Hopskipdrive,Los Angeles,8.0,2020-03-13,10.0,Transportat…,https://layoffs.fyi/2020/04/02/ hopskipdrive-l...,Unknown,45.0,United States,False,False,False,2020,3,1,2020Q1
2,Help.Com,Austin,16.0,2020-03-16,100.0,Support,LinkedIn,Seed,6.0,United States,False,False,False,2020,3,1,2020Q1
3,Service,Los Angeles,109.0,2020-03-16,100.0,Travel,https://techcrunch.com/2020/ 03/16/travel-savi...,Seed,5.0,United States,False,True,False,2020,3,1,2020Q1
4,Inspirato,Denver,130.0,2020-03-16,22.0,Travel,https://businessden.com/2020 /03/16/inspirato-...,Series C,79.0,United States,False,False,False,2020,3,1,2020Q1


In [18]:
df.to_csv("../data/cleaned_layoff_dataset.csv", index=False)
print("Cleaned dataset exported to ../data/cleaned_layoff_dataset.csv ✅")

Cleaned dataset exported to ../data/cleaned_layoff_dataset.csv ✅
